In [1]:
import torch
import torch.nn as nn
from transformers import AutoTokenizer, AutoModelForCausalLM

device="cuda"

In [4]:
model_id = "gpt2"
gpt2_tokenizer = AutoTokenizer.from_pretrained(model_id)
gpt2 = AutoModelForCausalLM.from_pretrained(model_id, device_map="auto", dtype="auto")

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

e:\Program\handson-mlp\.venv\Lib\site-packages\huggingface_hub\file_download.py:137: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\User\.cache\huggingface\hub\models--gpt2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [6]:
from transformers import PreTrainedModel, PreTrainedTokenizerBase

def generate(
    model: PreTrainedModel, tokenizer: PreTrainedTokenizerBase, prompt: str, 
    max_new_tokens: int = 50, **generate_kwargs,) -> str:
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(**inputs, max_new_tokens=max_new_tokens,
    pad_token_id=tokenizer.eos_token_id, **generate_kwargs)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [7]:
prompt = "Scientists found a talking unicorn today. Here's the full story:"
generate(gpt2, gpt2_tokenizer, prompt)

"Scientists found a talking unicorn today. Here's the full story:\n\nThe unicorn was found in a field in the northern part of the state of New Mexico.\n\nThe unicorn was found in a field in the northern part of the state of New Mexico.\n\nThe unicorn was found in a field in"

In [8]:
torch.manual_seed(42)
generate(gpt2, gpt2_tokenizer, prompt, do_sample=True)

"Scientists found a talking unicorn today. Here's the full story:\n\nThere aren't lots of other unicorns and they have been making their way across the United States since at least the 1800s, but this year there weren't a solitary unicorn on the land. Today, there are around 1,000."

In [9]:
torch.manual_seed(42)
generate(gpt2, gpt2_tokenizer, prompt, do_sample=True, top_p=0.6)

"Scientists found a talking unicorn today. Here's the full story:\n\nThe unicorn is an amphibian found in the Middle East, and it is not known where it came from.\n\nThe researchers said they discovered the unicorn in Egypt in a cave in the desert of the Sinai Peninsula.\n\nThe researchers"

In [11]:
DEFAULT_TEMPLATE = "Capital city of France = Paris\nCapital city of {country} ="

def get_capital_city(model:PreTrainedModel, tokenizer:PreTrainedTokenizerBase, country:str, template=DEFAULT_TEMPLATE):
    prompt = template.format(country=country)
    extended_text = generate(model, tokenizer, prompt, max_new_tokens=50)
    answer = extended_text[len(prompt):]
    return answer.strip().splitlines()[0].strip()

In [12]:
get_capital_city(gpt2, gpt2_tokenizer, "United Kingdom")

'London'

In [13]:
get_capital_city(gpt2, gpt2_tokenizer, "Mexico")

'Mexico City'